# Evaluasi Dampak Slang Normalization dan N-gram Feature terhadap Performa Model Boosting
## (XGBoost, LightGBM, CatBoost, AdaBoost) Dalam Deteksi Spam

**Authors:** Alexander Christian Suryanto Linggodigdo, Fanny Octaviana Pangestu, Howard Frelindo Goh  
**Institution:** Computer Science – BINUS University

## 1. Setup & Imports

In [ ]:
import re
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions

from slang_dict import SLANG_DICT

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('dataset.csv', encoding='latin-1', usecols=[0, 1], header=0)
df.columns = ['label', 'text']
df.dropna(subset=['label', 'text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nClass balance: {df['label'].value_counts(normalize=True).round(3).to_dict()}")
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

df['text_len'] = df['text'].apply(len)
for label, color in [('ham', 'steelblue'), ('spam', 'tomato')]:
    axes[1].hist(df[df['label'] == label]['text_len'], bins=40, alpha=0.6, color=color, label=label)
axes[1].set_title('Message Length Distribution')
axes[1].set_xlabel('Character Length')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150)
plt.show()

## 3. Text Preprocessing Pipeline

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Build regex pattern for slang replacement (longest match first to avoid partial replacement)
_slang_keys_sorted = sorted(SLANG_DICT.keys(), key=len, reverse=True)
_slang_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _slang_keys_sorted) + r')\b'
)

def preprocess(text: str, normalize_slang: bool = True) -> str:
    """Full preprocessing pipeline as described in Section 3.1."""
    # Step 1 – Case folding
    text = text.lower()

    # Step 2 – Entity replacement (URL, phone number)
    text = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', text)
    text = re.sub(r'\b(?:\+?\d[\d\s\-().]{7,})\b', ' phonetoken ', text)

    # Step 3 – Contraction expansion
    text = contractions.fix(text)

    # Step 4 – Punctuation removal (keep alphanumeric + spaces)
    text = re.sub(r"[^a-z0-9\s]", ' ', text)

    # Step 5 – Slang normalization (optional, using word boundaries)
    if normalize_slang:
        text = _slang_pattern.sub(lambda m: SLANG_DICT[m.group(0)], text)

    # Step 6 – Tokenization
    tokens = text.split()

    # Step 7 – Stopword & standalone number removal
    tokens = [t for t in tokens if t not in STOP_WORDS and not t.isdigit()]

    # Step 8 – Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)


# Quick sanity check
test_samples = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Txt FA to 87121",
    "U dun say so early hor... U c already then say...",
    "Congrats! Ur the winner of gr8 prize! Claim ur reward at http://win.com",
]
print("=== Preprocessing Samples ===")
for s in test_samples:
    print(f"  ORIGINAL : {s}")
    print(f"  NO SLANG : {preprocess(s, normalize_slang=False)}")
    print(f"  WITH NORM: {preprocess(s, normalize_slang=True)}")
    print()

In [ ]:
print('Applying preprocessing (this may take a moment)...')

t0 = time.time()
df['processed_no_norm'] = df['text'].apply(lambda x: preprocess(x, normalize_slang=False))
df['processed_with_norm'] = df['text'].apply(lambda x: preprocess(x, normalize_slang=True))
print(f'Done in {time.time()-t0:.1f}s')

# Encode labels: spam=1, ham=0
le = LabelEncoder()
y = le.fit_transform(df['label'])  # ham=0, spam=1
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

## 4. Feature Extraction – TF-IDF with N-gram Variants

In [ ]:
NGRAM_CONFIGS = {
    'Unigram': (1, 1),
    'Bigram':  (1, 2),
    'Trigram': (1, 3),
}

NORM_CONFIGS = {
    'No Normalization':   df['processed_no_norm'],
    'With Normalization': df['processed_with_norm'],
}

# Train/test split indices (shared across all experiments)
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
y_train, y_test = y[idx_train], y[idx_test]

print(f'Train size: {len(idx_train)} | Test size: {len(idx_test)}')
print(f'Train spam ratio: {y_train.mean():.3f} | Test spam ratio: {y_test.mean():.3f}')

# Pre-compute all feature matrices
feature_matrices = {}
for norm_name, corpus in NORM_CONFIGS.items():
    for ngram_name, ngram_range in NGRAM_CONFIGS.items():
        key = (norm_name, ngram_name)
        vec = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=10000,
            sublinear_tf=True
        )
        X_tr = vec.fit_transform(corpus.iloc[idx_train])
        X_te = vec.transform(corpus.iloc[idx_test])
        feature_matrices[key] = (X_tr, X_te, vec)
        print(f'[{norm_name} | {ngram_name}] vocab={len(vec.vocabulary_):,} | train={X_tr.shape} | test={X_te.shape}')

## 5. Model Training & Evaluation

In [ ]:
def get_models():
    return {
        'AdaBoost': AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1),
            n_estimators=100,
            learning_rate=1.0,
            random_state=RANDOM_STATE
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100,
            num_leaves=31,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            verbose=-1
        ),
        'CatBoost': CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            random_seed=RANDOM_STATE,
            verbose=0
        )
    }


def evaluate(model, X_tr, X_te, y_tr, y_te):
    t_start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t_start

    y_pred = model.predict(X_te)
    try:
        y_prob = model.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_prob = y_pred.astype(float)

    return {
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall':    recall_score(y_te, y_pred, zero_division=0),
        'F1-Score':  f1_score(y_te, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_te, y_prob),
        'Train Time (s)': round(train_time, 2),
        '_y_pred': y_pred,
        '_y_prob': y_prob,
    }

print('Model evaluation functions defined.')

In [ ]:
results = []  # list of dicts for DataFrame
raw_results = {}  # (norm, ngram, model) -> result dict with predictions

total = len(NORM_CONFIGS) * len(NGRAM_CONFIGS) * 4
done = 0

for norm_name in NORM_CONFIGS:
    for ngram_name in NGRAM_CONFIGS:
        X_tr, X_te, _ = feature_matrices[(norm_name, ngram_name)]
        models = get_models()
        for model_name, model in models.items():
            done += 1
            print(f'[{done}/{total}] {norm_name} | {ngram_name} | {model_name}...', end=' ', flush=True)
            res = evaluate(model, X_tr, X_te, y_train, y_test)
            raw_results[(norm_name, ngram_name, model_name)] = res
            results.append({
                'Normalization': norm_name,
                'N-gram': ngram_name,
                'Model': model_name,
                'Precision': res['Precision'],
                'Recall': res['Recall'],
                'F1-Score': res['F1-Score'],
                'ROC-AUC': res['ROC-AUC'],
                'Train Time (s)': res['Train Time (s)'],
            })
            print(f"F1={res['F1-Score']:.4f} | AUC={res['ROC-AUC']:.4f} | {res['Train Time (s)']}s")

results_df = pd.DataFrame(results)
print('\nAll experiments complete.')

## 6. Results Summary

In [ ]:
display_df = results_df.copy()
for col in ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    display_df[col] = display_df[col].map('{:.4f}'.format)

pd.set_option('display.max_rows', 100)
display_df

In [ ]:
# Best configuration by F1-Score
best = results_df.loc[results_df['F1-Score'].idxmax()]
print('=== Best Configuration (by F1-Score) ===')
print(best.to_string())

# Best per model
print('\n=== Best F1-Score per Model ===')
print(results_df.groupby('Model')['F1-Score'].max().sort_values(ascending=False))

In [ ]:
results_df.to_csv('experiment_results.csv', index=False)
print('Results saved to experiment_results.csv')

## 7. Visualizations

In [ ]:
# ─── Fig 1: F1-Score heatmap per Normalization × N-gram × Model ───
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, norm in zip(axes, NORM_CONFIGS.keys()):
    sub = results_df[results_df['Normalization'] == norm]
    pivot = sub.pivot(index='Model', columns='N-gram', values='F1-Score')
    # Reorder columns
    pivot = pivot[['Unigram', 'Bigram', 'Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
                vmin=results_df['F1-Score'].min() * 0.97,
                vmax=results_df['F1-Score'].max() * 1.01)
    ax.set_title(f'F1-Score — {norm}', fontsize=13)

plt.suptitle('F1-Score Heatmap: Models × N-gram × Normalization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Fig 2: ROC-AUC heatmap ───
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, norm in zip(axes, NORM_CONFIGS.keys()):
    sub = results_df[results_df['Normalization'] == norm]
    pivot = sub.pivot(index='Model', columns='N-gram', values='ROC-AUC')
    pivot = pivot[['Unigram', 'Bigram', 'Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='Blues', ax=ax,
                vmin=results_df['ROC-AUC'].min() * 0.97,
                vmax=1.0)
    ax.set_title(f'ROC-AUC — {norm}', fontsize=13)

plt.suptitle('ROC-AUC Heatmap: Models × N-gram × Normalization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_auc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Fig 3: Grouped bar chart – impact of normalization (Unigram, best model) ───
metrics = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC']
models_list = list(get_models().keys())

fig, axes = plt.subplots(1, len(NGRAM_CONFIGS), figsize=(18, 5), sharey=False)
width = 0.35
x = np.arange(len(models_list))

for ax, ngram_name in zip(axes, NGRAM_CONFIGS.keys()):
    f1_no   = [results_df[(results_df['Normalization']=='No Normalization') &
                           (results_df['N-gram']==ngram_name) &
                           (results_df['Model']==m)]['F1-Score'].values[0] for m in models_list]
    f1_with = [results_df[(results_df['Normalization']=='With Normalization') &
                           (results_df['N-gram']==ngram_name) &
                           (results_df['Model']==m)]['F1-Score'].values[0] for m in models_list]
    bars1 = ax.bar(x - width/2, f1_no,   width, label='No Normalization', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, f1_with, width, label='With Normalization', color='tomato', alpha=0.8)
    ax.set_xlabel('Model')
    ax.set_ylabel('F1-Score')
    ax.set_title(f'F1-Score: {ngram_name}')
    ax.set_xticks(x)
    ax.set_xticklabels(models_list, rotation=15)
    ax.set_ylim(0.85, 1.01)
    ax.legend()
    # Annotate bars
    for bar in bars1:
        ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.suptitle('F1-Score: Impact of Slang Normalization per N-gram & Model', fontsize=13)
plt.tight_layout()
plt.savefig('bar_normalization_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Fig 4: Confusion matrices for best configuration ───
best_row = results_df.loc[results_df['F1-Score'].idxmax()]
best_norm  = best_row['Normalization']
best_ngram = best_row['N-gram']
best_model = best_row['Model']

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, model_name in zip(axes, models_list):
    key = (best_norm, best_ngram, model_name)
    y_pred = raw_results[key]['_y_pred']
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{model_name}\n({best_norm[:4]}… | {best_ngram})', fontsize=10)

plt.suptitle(f'Confusion Matrices — Best Config: {best_norm} | {best_ngram}', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Fig 5: ROC Curves for best configuration ───
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'tomato', 'green', 'purple']

for model_name, color in zip(models_list, colors):
    key = (best_norm, best_ngram, model_name)
    y_prob = raw_results[key]['_y_prob']
    auc = raw_results[key]['ROC-AUC']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {best_norm} | {best_ngram}')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ─── Fig 6: Training time comparison ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, norm_name in zip(axes, NORM_CONFIGS.keys()):
    sub = results_df[results_df['Normalization'] == norm_name]
    pivot = sub.pivot(index='Model', columns='N-gram', values='Train Time (s)')
    pivot = pivot[['Unigram', 'Bigram', 'Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Greens', ax=ax)
    ax.set_title(f'Training Time (s) — {norm_name}')

plt.suptitle('Training Time Heatmap (seconds)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_traintime.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Normalization Impact Analysis

In [ ]:
print('=== F1-Score Delta: With Normalization vs No Normalization ===')
delta_rows = []
for ngram_name in NGRAM_CONFIGS:
    for model_name in models_list:
        f1_no   = results_df[(results_df['Normalization']=='No Normalization') &
                              (results_df['N-gram']==ngram_name) &
                              (results_df['Model']==model_name)]['F1-Score'].values[0]
        f1_with = results_df[(results_df['Normalization']=='With Normalization') &
                              (results_df['N-gram']==ngram_name) &
                              (results_df['Model']==model_name)]['F1-Score'].values[0]
        delta = f1_with - f1_no
        delta_rows.append({'N-gram': ngram_name, 'Model': model_name,
                           'F1 (No Norm)': round(f1_no, 4),
                           'F1 (With Norm)': round(f1_with, 4),
                           'Delta': round(delta, 4)})

delta_df = pd.DataFrame(delta_rows)
delta_df['Direction'] = delta_df['Delta'].apply(lambda d: '▲ Improved' if d > 0 else ('▼ Degraded' if d < 0 else '= Same'))
display(delta_df)

avg_delta = delta_df['Delta'].mean()
print(f'\nAverage F1 delta across all experiments: {avg_delta:+.4f}')

## 9. Final Summary Table

In [ ]:
print('=== Top 10 Configurations by F1-Score ===')
top10 = results_df.nlargest(10, 'F1-Score')[['Normalization','N-gram','Model','Precision','Recall','F1-Score','ROC-AUC','Train Time (s)']]
top10_display = top10.copy()
for col in ['Precision','Recall','F1-Score','ROC-AUC']:
    top10_display[col] = top10_display[col].map('{:.4f}'.format)
display(top10_display)

print('\n=== Best Model per Algorithm (Overall F1-Score) ===')
best_per_model = results_df.loc[results_df.groupby('Model')['F1-Score'].idxmax()]\
    [['Model','Normalization','N-gram','F1-Score','ROC-AUC']].sort_values('F1-Score', ascending=False)
display(best_per_model.reset_index(drop=True))

print('\n=== Success Criteria Evaluation ===')
best_f1 = results_df['F1-Score'].max()
baseline_f1 = results_df[(results_df['Normalization']=='No Normalization') &
                          (results_df['N-gram']=='Unigram')]['F1-Score'].mean()
print(f'  Baseline avg F1 (No Norm, Unigram): {baseline_f1:.4f}')
print(f'  Best F1 overall:                    {best_f1:.4f}')
print(f'  Improvement:                        {best_f1-baseline_f1:+.4f}')

fp_col = []
for model_name in models_list:
    key = (best_norm, best_ngram, model_name)
    y_pred = raw_results[key]['_y_pred']
    cm = confusion_matrix(y_test, y_pred)
    fp = cm[0][1]  # ham predicted as spam
    fn = cm[1][0]  # spam predicted as ham
    fp_col.append({'Model': model_name, 'FP (Ham→Spam)': fp, 'FN (Spam→Ham)': fn})
print(f'\nFalse Positive/Negative summary ({best_norm} | {best_ngram}):')
display(pd.DataFrame(fp_col))

In [ ]:
print('\n=== Experiment Complete ===')
print(f'Results saved to: experiment_results.csv')
print('Figures saved:    eda_distribution.png, heatmap_f1.png, heatmap_auc.png,')
print('                  bar_normalization_impact.png, confusion_matrices.png,')
print('                  roc_curves.png, heatmap_traintime.png')